In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import scipy as sp
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from statsmodels.stats.multitest import multipletests

from src.notebook import load_notebook_data, data_dict_to_dataframe
from src.plotting import plot_repeated_measures_violins, plot_subject_level_violins
from src.timescales import create_ints_dataframe_custom_grouping, convert_data_dict, average_data_dict_converted
from snaplab_tools.plotting.plotting import plot_correlation_unity, annotate_significance_brackets
from snaplab_tools.plotting.utils import get_my_colors
from snaplab_tools.stats import paired_ttest_vs_reference, decoupling_test, significance_stars
from snaplab_tools.utils import winsorize

In [ ]:
nas_scratch_base = '/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales'

In [ ]:
my_colors = get_my_colors(cat_trio=False, as_list=True)

brain_map = 'SA_Axis'

# Maps raw condition keys -> display labels
LABEL_MAP = {
    'rest':             'Rest',
    'reversallearning': 'RL',
    'frac2back':        'n-back',
    'nback':            'n-back',
    'task-Rest':        'Rest',
    'task-Movies':      'Movies',
    'task-Tasks':       'Tasks',
    'tfMRIEMOTIONPA':   'Emotion',
    'tfMRICARITPA':     'Carit',
    'tfMRIGUESSINGPA':  'Guessing',

}

# Per-dataset loading configuration
DATASET_CONFIGS = {
    'HCP-YA': {
        'which_modality': 'FMRI',
        'use_7t':         True,
        'use_resids':     True,
        'output_type':    'timescale_auc_zero_crossing_bp-0.01-0.08hz',
        'collapse_all':   True,
        'custom_order':   ['task-Rest', 'task-Movies', 'task-Tasks'],
        'title':          'HCP-YA',
    },
    'WAND': {
        'which_modality': 'FMRI',
        'use_7t':         False,
        'use_resids':     False,
        'output_type':    'timescale_auc_zero_crossing',
        'collapse_all':   False,
        'custom_order':   ['rest', 'reversallearning'],
        'title':          'WAND',
    },
    'HCP-D': {
        'which_modality': 'FMRI',
        'use_7t':         False,
        'use_resids':     True,
        'output_type':    'timescale_auc_zero_crossing_bp-0.01-0.08hz',
        # 'collapse_all':   True,
        # 'custom_order':   ['task-Rest', 'task-Tasks'],
        'collapse_all':   False,
        'custom_order':   ['task-Rest', 'tfMRIEMOTIONPA', 'tfMRICARITPA', 'tfMRIGUESSINGPA'],
        'title':          'HCP-D',
    },
    'Penn_LEAD': {
        'which_modality': 'FMRI',
        'use_7t':         False,
        'use_resids':     True,
        'output_type':    'timescale_auc_zero_crossing',
        'collapse_all':   False,
        'custom_order':   ['rest', 'nback'],
        'title':          'Penn LEAD',
    },
}

In [ ]:
brain_map = 'SA_Axis'
all_data = {}

# Penn LEAD has two sessions per subject. When True, drop all session-2 scans
# immediately after loading so every downstream figure and test (Fig 1 rows,
# decoupling test, etc.) uses session 1 only.
PENN_SESSION1_ONLY = True

for dataset, cfg in DATASET_CONFIGS.items():
    print(f'\n=== Loading {dataset} ===')

    notebook_data = load_notebook_data(
        dataset=dataset,
        which_modality=cfg['which_modality'],
        use_7t=cfg['use_7t'],
        output_type=cfg['output_type'],
        use_resids=cfg['use_resids'],
        load_subject_data=False,
    )

    timescales_processor = notebook_data['timescales_processor']
    data_dict = notebook_data['data_dict']
    df_neuro  = notebook_data['supporting_data']['df_neuro']

    # Optionally restrict Penn LEAD to session 1: drop session-2 scans from both the
    # data_dict and the scan_registry so all downstream grouping/plotting uses ses-1.
    if PENN_SESSION1_ONLY and dataset == 'Penn_LEAD':
        keep_scans = [s for s in timescales_processor.scan_registry if str(s.session) == '1']
        keep_keys  = {str(s) for s in keep_scans}
        data_dict  = {k: v for k, v in data_dict.items() if k in keep_keys}
        timescales_processor.scan_registry = keep_scans
        n_kept_sub = len({s.subject_id for s in keep_scans})
        print(f'  Penn LEAD: session 1 only -> {len(keep_scans)} scans, {n_kept_sub} subjects')

    # Scan-grouping shared across df_ints / centiles / per-region arrays.
    # Rest is always collapsed; HCP-YA additionally collapses 7T movies and 3T tasks.
    group_kwargs = (
        dict(average_all_rest=True, average_7t_movies=True, average_3t_tasks=True)
        if cfg['collapse_all'] else
        dict(average_all_rest=True)
    )

    # --- df_ints: regional mean INTs per condition ---
    df_ints = create_ints_dataframe_custom_grouping(
        data_dict, sort_by_median=not cfg['collapse_all'], **group_kwargs,
    )
    if cfg['custom_order']:
        df_ints = df_ints[cfg['custom_order']]

    # --- data_dict_centiles: subject × SA-centile arrays per condition (Fig 1 panel C) ---
    data_dict_centiles = convert_data_dict(
        data_dict, timescales_processor.scan_registry,
        brain_map=df_neuro[brain_map], verbose=False,
    )
    data_dict_centiles = average_data_dict_converted(data_dict_centiles, **group_kwargs)
    if cfg['custom_order']:
        data_dict_centiles = {k: data_dict_centiles[k] for k in cfg['custom_order'] if k in data_dict_centiles}

    # --- data_dict_converted: subject × region INT arrays per condition (unbinned) ---
    # Same grouping as above but keeping all 400 regions, for the decoupling test.
    data_dict_converted = convert_data_dict(
        data_dict, timescales_processor.scan_registry, verbose=False,
    )
    data_dict_converted = average_data_dict_converted(data_dict_converted, **group_kwargs)
    if cfg['custom_order']:
        data_dict_converted = {k: data_dict_converted[k] for k in cfg['custom_order'] if k in data_dict_converted}
    subject_ids = list(dict.fromkeys(scan.subject_id for scan in timescales_processor.scan_registry))

    all_data[dataset] = {
        'df_ints':             df_ints,
        'df_neuro':            df_neuro,
        'data_dict_centiles':  data_dict_centiles,
        'data_dict_converted': data_dict_converted,
        'subject_ids':         subject_ids,
        'cfg':                 cfg,
    }

    # HCP-YA supplementary figure also needs the raw (un-grouped) inputs.
    if dataset == 'HCP-YA':
        all_data[dataset]['data_dict']     = data_dict
        all_data[dataset]['scan_registry'] = timescales_processor.scan_registry

    print(f'  df_ints columns: {df_ints.columns.tolist()}')
    print(f'  centiles keys:   {list(data_dict_centiles.keys())}')
    print(f'  converted shapes: {[(k, v.shape) for k, v in data_dict_converted.items()]}')

# Figure 1

In [ ]:
dataset_order = list(all_data.keys())
col_widths    = [len(all_data[d]['df_ints'].columns) for d in dataset_order]
total_conds   = sum(col_widths)

fig = plt.figure(figsize=(8.5, 3.75), dpi=300)
gs  = gridspec.GridSpec(
    2, len(dataset_order),
    width_ratios=col_widths,
    hspace=0.08, wspace=0.4,
)

ax0_first = ax1_first = None

for col_idx, dataset in enumerate(dataset_order):
    d   = all_data[dataset]
    cfg = d['cfg']
    is_first_col = (col_idx == 0)

    df_ints            = d['df_ints'].copy()
    df_neuro           = d['df_neuro']
    data_dict_centiles = d['data_dict_centiles']

    ax0 = fig.add_subplot(gs[0, col_idx])
    ax1 = fig.add_subplot(gs[1, col_idx])
    if is_first_col:
        ax0_first, ax1_first = ax0, ax1

    # Build centile df and apply WAND-specific winsorize
    df_centiles = data_dict_to_dataframe(data_dict_centiles)
    if dataset == 'WAND' and 'reversallearning' in df_centiles.columns:
        df_centiles['reversallearning'] = winsorize(df_centiles['reversallearning'])

    # Rename columns to display labels
    df_ints     = df_ints.rename(columns=LABEL_MAP)
    df_centiles = df_centiles.rename(columns=LABEL_MAP)

    # Align centile column order to match df_ints
    cond_cols = [c for c in df_centiles.columns if c != 'subject']
    ordered   = [c for c in df_ints.columns if c in cond_cols]
    df_centiles = df_centiles[['subject'] + ordered]

    # --- Row A: regional mean violins ---
    ax0 = plot_repeated_measures_violins(
        df_ints, ax0,
        color_vector=df_neuro[brain_map], colormap='sydnor_sa',
        add_colorbar=False,
        point_alpha=1, line_alpha=0.3, point_size=0.25, line_width=0.25,
        violin_alpha=0.25, correlation_method='spearman', annotate_variance=False,
    )
    ax0.set_xticklabels([])
    ax0.set_xlabel('')
    ax0.set_title(cfg['title'], fontsize=8, fontweight='bold')

    # Paired t-tests (regions paired) vs Rest: are INTs lower in task/movies than rest?
    # Rest is the reference (leftmost column); brackets stack Rest -> each condition.
    ttest_vs_rest = paired_ttest_vs_reference(
        df_ints, reference=df_ints.columns[0],
        alternative='two-sided', correction='fdr_bh',
    )
    print(ttest_vs_rest)
    sig_pairs  = [(0, i) for i in range(1, len(df_ints.columns))]
    sig_labels = [ttest_vs_rest.loc[c, 'sig'] for c in df_ints.columns[1:]]
    annotate_significance_brackets(ax0, sig_pairs, sig_labels, fontsize=6, text_pad_frac=-0.005)

    # --- Row B: subject-level centile scatter ---
    ax1, _ = plot_subject_level_violins(
        df_centiles, ax1, np.arange(10),
        point_alpha=0.75, point_size=0.25,
        show_violins=False, show_lines=False, fit_trends=True,
        show_r2_annotation=False, annotate_variance=False,
        hide_legend=True, sort_by_median=False,
    )
    ax1.set_xlabel('')

    # y-labels only on leftmost column
    if is_first_col:
        ax0.set_ylabel('INTs (s, mean)',     fontsize=8)
        ax1.set_ylabel('INTs (s, subjects)', fontsize=8)
    else:
        ax0.set_ylabel('')
        ax1.set_ylabel('')

    for ax in [ax0, ax1]:
        ax.tick_params(axis='y', which='major', pad=1)
        ax.tick_params(axis='x', which='major', pad=1)
        ax.yaxis.set_label_coords(-0.2, 0.5)

plt.tight_layout()

fig.savefig(
    os.path.join(nas_scratch_base, 'timescales_analysis_fig1.png'),
    bbox_inches='tight', dpi=1000, transparent=True,
)
plt.show()

# Rest→task decoupling of INTs from the S-A axis

In [ ]:
# Formal within-dataset test of the rest->task decoupling of INTs from the S-A axis.
# For each dataset x non-rest condition, test whether each subject's S-A coupling
# (Spearman rho of regional INTs vs the S-A axis, Fisher-z transformed) is lower
# during the condition than at rest, via a paired t-test. Reuses the per-subject,
# per-region INT arrays (data_dict_converted) and the S-A axis (df_neuro[brain_map])
# already loaded for Figure 1 panel C.
#
# Rows are matched within a dataset (paired design). For HCP-YA / WAND / HCP-D each
# row is a subject; for Penn LEAD each row is a scan, so with PENN_SESSION1_ONLY the
# unit is effectively one per subject (session 1), else two scans per subject.
# decoupling_test drops any row lacking usable data (or with an undefined coupling)
# in either condition; `n` reports the subjects/scans actually used per comparison.

decoupling_rows = []
for dataset, d in all_data.items():
    conds    = d['cfg']['custom_order']
    rest_key = conds[0]
    sa_axis  = d['df_neuro'][brain_map].to_numpy()
    int_rest = d['data_dict_converted'][rest_key]

    for cond_key in conds[1:]:
        int_cond = d['data_dict_converted'][cond_key]
        rho_rest, rho_cond, t, p, n = decoupling_test(int_rest, int_cond, sa_axis)
        decoupling_rows.append({
            'dataset':       d['cfg']['title'],
            'condition':     LABEL_MAP.get(cond_key, cond_key),
            'n':             n,
            'rho_rest':      rho_rest,
            'rho_condition': rho_cond,
            't':             t,
            'p':             p,
        })

df_decoupling = pd.DataFrame(
    decoupling_rows,
    columns=['dataset', 'condition', 'n', 'rho_rest', 'rho_condition', 't', 'p'],
)
# Benjamini-Hochberg FDR across all valid dataset x condition comparisons.
df_decoupling['p_fdr'] = np.nan
valid = df_decoupling['p'].notna()
df_decoupling.loc[valid, 'p_fdr'] = multipletests(
    df_decoupling.loc[valid, 'p'].to_numpy(), method='fdr_bh',
)[1]
# Significance stars (***/**/*/ns) on the FDR-corrected p-values.
df_decoupling['sig'] = df_decoupling['p_fdr'].apply(significance_stars)

with pd.option_context('display.float_format', lambda v: f'{v:.3g}'):
    print(df_decoupling.to_string(index=False))

# Supplementary Figures: HCP-YA only

In [ ]:
# Supplementary Figure: HCP-YA Figure 1 without averaging over (task/movie) scans.
# Rest is collapsed: all 3T rest runs -> one column, all 7T rest runs -> one column.
# Movie and task scans are kept as individual columns, with phase-encoding directions
# still averaged (each 3T task's LR/RL -> a single column; 7T movies are single-PE).

SORT_BY_MEDIAN = True   # order columns by descending median regional-mean INT (as in Fig 1)

nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/HCP-YA'
os.makedirs(nas_scratch_dir, exist_ok=True)

hcpya        = all_data['HCP-YA']
data_dict_ya = hcpya['data_dict']
scan_reg_ya  = hcpya['scan_registry']
df_neuro_ya  = hcpya['df_neuro']

# --- Row A: regional-mean INTs, one column per scan ---
df_ints_scans = create_ints_dataframe_custom_grouping(
    data_dict_ya,
    average_3t_rest=True,         # all 3T rest runs -> Rest (3T)
    average_7t_rest=True,         # all 7T rest runs -> Rest (7T)
    average_3t_task_pairs=True,   # each task's LR/RL -> a single column
)

# --- Row B: subject x SA-centile arrays, one column per scan ---
centiles_scans = convert_data_dict(
    data_dict_ya, scan_reg_ya, brain_map=df_neuro_ya[brain_map], verbose=False,
)
centiles_scans = average_data_dict_converted(
    centiles_scans, average_3t_rest=True, average_7t_rest=True, average_3t_task_pairs=True,
)

# Display labels for the individual scans
SUPP_LABEL_MAP = {
    'task-REST_3T':    'Rest (3T)',
    'task-REST_7T':    'Rest (7T)',
    'tfMRIMOVIE17TAP': 'Movie1',
    'tfMRIMOVIE27TPA': 'Movie2',
    'tfMRIMOVIE37TPA': 'Movie3',
    'tfMRIMOVIE47TAP': 'Movie4',
    'task-WM':         'WM',
    'task-EMOTION':    'Emotion',
    'task-GAMBLING':   'Gambling',
    'task-LANGUAGE':   'Language',
    'task-RELATIONAL': 'Relational',
    'task-SOCIAL':     'Social',
}

# Column order: derived once and applied to BOTH rows so they stay aligned.
# (A fixed reindex here is what previously overrode create_ints' sort_by_median.)
present_keys = [k for k in SUPP_LABEL_MAP if k in df_ints_scans.columns]
if SORT_BY_MEDIAN:
    supp_order = (
        df_ints_scans[present_keys].median(axis=0)
        .sort_values(ascending=False).index.tolist()
    )
else:
    supp_order = present_keys
print(f'Supplementary scan columns ({len(supp_order)}): {supp_order}')

# Order + rename Row A
df_ints_scans = df_ints_scans[supp_order].rename(columns=SUPP_LABEL_MAP)

# Order + rename Row B (same order as Row A)
df_centiles_scans = data_dict_to_dataframe(centiles_scans)
cond_cols = [c for c in df_centiles_scans.columns if c != 'subject']
df_centiles_scans = df_centiles_scans[['subject'] + [c for c in supp_order if c in cond_cols]]
df_centiles_scans = df_centiles_scans.rename(columns=SUPP_LABEL_MAP)

# --- Figure ---
n_conds = len(df_ints_scans.columns)
fig = plt.figure(figsize=(0.5 * n_conds + 0.5, 4), dpi=300)
gs  = gridspec.GridSpec(2, 1, hspace=0.08)

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0])

# --- Row A: regional mean violins ---
ax0 = plot_repeated_measures_violins(
    df_ints_scans, ax0,
    color_vector=df_neuro_ya[brain_map], colormap='sydnor_sa',
    add_colorbar=False,
    point_alpha=1, line_alpha=0.3, point_size=0.25, line_width=0.25,
    violin_alpha=0.25, correlation_method='spearman', annotate_variance=False,
)
ax0.set_xticklabels([])
ax0.set_xlabel('')
ax0.set_title('HCP-YA (individual scans)', fontsize=8, fontweight='bold')
ax0.set_ylabel('INTs (s, mean)', fontsize=8)

# --- Row B: subject-level centile scatter ---
# sort_by_median stays False here: columns are already ordered above, and letting this
# re-sort by its own (centile) median would de-align Row B from Row A.
ax1, _ = plot_subject_level_violins(
    df_centiles_scans, ax1, np.arange(10),
    point_alpha=0.75, point_size=0.25,
    show_violins=False, show_lines=False, fit_trends=True,
    show_r2_annotation=False, annotate_variance=False,
    hide_legend=True, sort_by_median=False,
)
ax1.set_xlabel('')
ax1.set_ylabel('INTs (s, subjects)', fontsize=8)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=8)

for ax in [ax0, ax1]:
    ax.tick_params(axis='y', which='major', pad=1)
    ax.tick_params(axis='x', which='major', pad=1)
    ax.yaxis.set_label_coords(-0.06, 0.5)

plt.tight_layout()
fig.savefig(
    os.path.join(nas_scratch_dir, 'timescales_analysis_fig1_hcpya_scans.png'),
    bbox_inches='tight', dpi=1000, transparent=True,
)
plt.show()

In [ ]:
# Compute per-subject SA-axis Pearson correlation for HCP-YA supplementary figure
hcpya = all_data['HCP-YA']
subject_ids_hcpya  = hcpya['subject_ids']
data_dict_conv     = hcpya['data_dict_converted']
df_neuro_hcpya     = hcpya['df_neuro']
n_subs_hcpya       = len(subject_ids_hcpya)

df_corr = pd.DataFrame(index=subject_ids_hcpya)
df_corr.index.name = 'subject_ids'
for int_col in data_dict_conv.keys():
    for i in tqdm(np.arange(n_subs_hcpya), desc=int_col):
        subject_id = subject_ids_hcpya[i]
        nan_mask = np.isnan(data_dict_conv[int_col][i])
        corr = sp.stats.pearsonr(
            data_dict_conv[int_col][i][~nan_mask],
            df_neuro_hcpya[brain_map][~nan_mask],
        )
        df_corr.loc[subject_id, int_col] = corr[0]

df_corr.head()

In [ ]:
nas_scratch_dir = os.path.join(nas_scratch_base, 'HCP-YA')
os.makedirs(nas_scratch_dir, exist_ok=True)

fig, axes = plt.subplots(nrows=1, ncols=2, sharex=False, sharey=False, figsize=(4, 2), dpi=300)
plot_correlation_unity(df_corr['task-Rest'], df_corr['task-Movies'], axes[0],
                       marginal_color_x=my_colors[0], marginal_color_y=my_colors[1],
                       show_marginals=True, show_correlation=True, correlation_type='pearson',
                       grid=False, size=5, fontsize=8)
plot_correlation_unity(df_corr['task-Rest'], df_corr['task-Tasks'], axes[1],
                       marginal_color_x=my_colors[0], marginal_color_y=my_colors[1],
                       show_marginals=True, show_correlation=True, correlation_type='pearson',
                       grid=False, size=5, fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(nas_scratch_dir, 'neuroanatomy_INTs_correlations_unity.svg'),
            bbox_inches='tight', dpi=1000, transparent=True)
plt.show()